In [1]:
import os
import numpy as np
import multiprocessing as mp

from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits

# ============================================================
# SETTINGS
# ============================================================

DATA_DIR = "fisher_data_all4"

OUTDIR = "Ppca_outputs"

os.makedirs(
    OUTDIR,
    exist_ok=True
)

LMAX = 5000
D = LMAX - 1

NPARAMS = 11

PCA_VAR = 0.995

EPS = 1e-12

NPROC = 4

# ============================================================
# LOAD RAW DATA
# ============================================================

print("\n================================================")
print("LOADING DATA")
print("================================================")

TT = np.fromfile(
    f"{DATA_DIR}/TT_4999.bin",
    dtype=np.float64
).reshape(-1, D)

EE = np.fromfile(
    f"{DATA_DIR}/EE_4999.bin",
    dtype=np.float64
).reshape(-1, D)

BB = np.fromfile(
    f"{DATA_DIR}/BB_4999.bin",
    dtype=np.float64
).reshape(-1, D)

TE = np.fromfile(
    f"{DATA_DIR}/TE_4999.bin",
    dtype=np.float64
).reshape(-1, D)

params = np.fromfile(
    f"{DATA_DIR}/params_4999.bin",
    dtype=np.float64
).reshape(-1, NPARAMS)

ells = np.fromfile(
    f"{DATA_DIR}/ells_4999.bin",
    dtype=np.int64
)

train_idx = np.fromfile(
    f"{DATA_DIR}/train_idx.bin",
    dtype=np.int64
)

test_idx = np.fromfile(
    f"{DATA_DIR}/test_idx.bin",
    dtype=np.int64
)

N = TT.shape[0]

print("\nShapes")

print("TT     :", TT.shape)
print("EE     :", EE.shape)
print("BB     :", BB.shape)
print("TE     :", TE.shape)
print("params :", params.shape)
print("ells   :", ells.shape)

print("\nTrain samples:", len(train_idx))
print("Test samples :", len(test_idx))

# ============================================================
# SAVE RAW SUPPORT FILES
# ============================================================

params.tofile(
    f"{OUTDIR}/params_4999.bin"
)

ells.astype(np.int64).tofile(
    f"{OUTDIR}/ells_4999.bin"
)

# ============================================================
# TRANSFORMS
# ============================================================

print("\n================================================")
print("TRANSFORMS")
print("================================================")

TT_t = np.log10(TT + EPS)

EE_t = np.log10(EE + EPS)

BB_t = np.log10(BB + EPS)

rho = TE / np.sqrt(
    (TT + EPS)
    *
    (EE + EPS)
)

rho = np.clip(
    rho,
    -0.9999,
    0.9999
)

RHO_t = np.arctanh(rho)

data_dict = {
    "TT": TT_t,
    "EE": EE_t,
    "BB": BB_t,
    "RHO": RHO_t
}

# ============================================================
# PCA WORKER
# ============================================================

def run_pca_worker(args):

    q, Y_all = args

    print("\n================================================")
    print(f"PCA : {q}")
    print("================================================")

    Y_train = Y_all[train_idx]
    Y_test  = Y_all[test_idx]

    print("\nInput shapes")

    print("Train:", Y_train.shape)
    print("Test :", Y_test.shape)

    # --------------------------------------------------------
    # Mean-center
    # --------------------------------------------------------

    mean_spec = Y_train.mean(axis=0)

    Y_train_c = Y_train - mean_spec
    Y_test_c  = Y_test  - mean_spec

    print("\nCentered stats")

    print("mean =", Y_train_c.mean())
    print("std  =", Y_train_c.std())

    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    with threadpool_limits(limits=1):

        pca = PCA(
            n_components=PCA_VAR,
            svd_solver='full'
        )

        coeff_train = pca.fit_transform(
            Y_train_c
        )

        coeff_test = pca.transform(
            Y_test_c
        )

    nmodes = coeff_train.shape[1]

    explained_var = np.sum(
        pca.explained_variance_ratio_
    )

    print("\nModes retained:", nmodes)

    print(
        "Explained variance:",
        explained_var
    )

    print("\nTop singular values")

    for i, s in enumerate(
        pca.singular_values_[:10]
    ):
        print(f"{i:2d}  {s:.12e}")

    # --------------------------------------------------------
    # Reconstruction
    # --------------------------------------------------------

    Y_rec_train = (
        pca.inverse_transform(coeff_train)
        +
        mean_spec
    )

    Y_rec_test = (
        pca.inverse_transform(coeff_test)
        +
        mean_spec
    )

    mse_train = np.mean(
        (Y_train - Y_rec_train) ** 2
    )

    mse_test = np.mean(
        (Y_test - Y_rec_test) ** 2
    )

    print(
        "\nTrain MSE:",
        mse_train
    )

    print(
        "Test MSE :",
        mse_test
    )

    # --------------------------------------------------------
    # Basis
    # --------------------------------------------------------

    basis = pca.components_.T

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    print("\nSaving outputs...")

    Y_train.tofile(
        f"{OUTDIR}/{q}_train_transformed.bin"
    )

    Y_test.tofile(
        f"{OUTDIR}/{q}_test_transformed.bin"
    )

    mean_spec.tofile(
        f"{OUTDIR}/{q}_mean.bin"
    )

    pca.singular_values_.tofile(
        f"{OUTDIR}/{q}_singular.bin"
    )

    basis.tofile(
        f"{OUTDIR}/{q}_basis.bin"
    )

    coeff_train.tofile(
        f"{OUTDIR}/{q}_coeff_train.bin"
    )

    coeff_test.tofile(
        f"{OUTDIR}/{q}_coeff_test.bin"
    )

    Y_rec_train.tofile(
        f"{OUTDIR}/{q}_recon_train.bin"
    )

    Y_rec_test.tofile(
        f"{OUTDIR}/{q}_recon_test.bin"
    )

    with open(
        f"{OUTDIR}/{q}_info.txt",
        "w"
    ) as f:

        f.write(
            f"Ntrain {len(train_idx)}\n"
        )

        f.write(
            f"Ntest {len(test_idx)}\n"
        )

        f.write(
            f"D {D}\n"
        )

        f.write(
            f"modes {nmodes}\n"
        )

        f.write(
            f"variance_cut {PCA_VAR:.15e}\n"
        )

        f.write(
            f"variance_loss "
            f"{1.0 - explained_var:.15e}\n"
        )

        f.write(
            f"explained_variance "
            f"{explained_var:.15e}\n"
        )

        f.write(
            f"mse_train "
            f"{mse_train:.15e}\n"
        )

        f.write(
            f"mse_test "
            f"{mse_test:.15e}\n"
        )

    print(f"\nFinished {q}")

# ============================================================
# MAIN PARALLEL EXECUTION
# ============================================================

if __name__ == "__main__":

    tasks = [
        ("TT", TT_t),
        ("EE", EE_t),
        ("BB", BB_t),
        ("RHO", RHO_t)
    ]

    with mp.Pool(processes=NPROC) as pool:

        pool.map(
            run_pca_worker,
            tasks
        )

    print("\n================================================")
    print("ALL PCA RUNS FINISHED")
    print("================================================")


LOADING DATA

Shapes
TT     : (1535, 4999)
EE     : (1535, 4999)
BB     : (1535, 4999)
TE     : (1535, 4999)
params : (1535, 11)
ells   : (4999,)

Train samples: 200
Test samples : 10

TRANSFORMS

PCA : TT

Input shapes
Train: (200, 4999)

================================================Test :
 PCA : EE(10, 4999)


Centered stats

Input shapesmean =
 Train:-2.45180728867885e-18 
(200, 4999)
std  =
PCA : BBTest :
  ================================================0.0005724033270831799
(10, 4999)


Input shapes

Centered statsTrain:
 mean =(200, 4999)
 -4.528339651200397e-18

Test :PCA : RHOstd  =  (10, 4999)

Centered stats0.0013873334874840903


Input shapes
mean =
 Train: -6.7914823150704716e-18(200, 4999)

Test : (10, 4999)
std  = 
Centered stats0.000594915131953487
mean = 
-2.758900884858173e-19
std  = 0.0005807495509962504

Modes retained: 7
Explained variance: 0.9952151043000909

Top singular values
 0  4.774843099510e-01
 1  2.511571659607e-01

Modes retained: 2  1.368048805667e-